In [ ]:
import pandas as pd
from sympy import categories

books = pd.read_csv("books_cleaned.csv")

In [2]:
category_mapping = {'Fiction' : "Fiction",
 'Juvenile Fiction': "Children's Fiction",
 'Biography & Autobiography': "Nonfiction",
 'History': "Nonfiction",
 'Literary Criticism': "Nonfiction",
 'Philosophy': "Nonfiction",
 'Religion': "Nonfiction",
 'Comics & Graphic Novels': "Fiction",
 'Drama': "Fiction",
 'Juvenile Nonfiction': "Children's Nonfiction",
 'Science': "Nonfiction",
 'Poetry': "Fiction"}

books["simple_categories"] = books["categories"].map(category_mapping)

In [7]:
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]

pipe = pipeline("zero-shot-classification",
                model="facebook/bart-large-mnli",
                device="cpu")

Device set to use cpu


In [8]:
sequence = books.loc[books["simple_categories"] == "Fiction","description"].reset_index(drop=True)[0]
pipe(sequence,fiction_categories)

{'sequence': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst

In [21]:
import numpy as np

def generate_predictions(sequence, categories):
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"])
    max_label = predictions["labels"][max_index]
    return max_label

In [22]:
isbns = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

In [32]:
from tqdm import tqdm

isbns = []
predicted_cats = []

# Filter rows with missing categories AND valid, non-empty descriptions
missing_cats = books.loc[
    books["simple_categories"].isna() & books["description"].notna() & (books["description"].str.strip() != ""),
    ["isbn13", "description"]
].reset_index(drop=True)

for i in tqdm(range(len(missing_cats))):
    try:
        sequence = missing_cats["description"][i]
        prediction = generate_predictions(sequence, fiction_categories)
        predicted_cats.append(prediction)
        isbns.append(missing_cats["isbn13"][i])
    except Exception as e:
        print(f" Error at index {i}: {e}")

100%|██████████| 1522/1522 [25:48<00:00,  1.02s/it]


In [34]:
missing_predicted_df = pd.DataFrame({
    "isbn13": isbns,
    "predicted_categories": predicted_cats
})

In [36]:
missing_predicted_df

,isbn13,predicted_categories
0,9780002261982,Fiction
1,9780006280897,Nonfiction
2,9780006280934,Nonfiction
3,9780006380832,Nonfiction
4,9780006470229,Fiction
...,...,...
1517,9788125026600,Nonfiction
1518,9788171565641,Fiction
1519,9788172235222,Fiction
1520,9788173031014,Nonfiction


In [37]:
books = pd.merge(books,missing_predicted_df,on="isbn13", how="left")
books["simple_categories"] = np.where(books["simple_categories"].isna(),books["predicted_categories"],books["simple_categories"])
books = books.drop(columns="predicted_categories")

In [38]:
books.to_csv("books_with_categories.csv", index=False)